In [5]:
%load_ext dotenv
%dotenv

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


In [6]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel

In [7]:
chat_template_books = ChatPromptTemplate.from_template(
    '''
    Suggest three of the best intermediate-level {programming language} books.
    Answer only by listing the books.
    '''
)

chat_template_projects = ChatPromptTemplate.from_template(
    '''
    Suggest three interesting {programming language} projects suitable for intermediate-level programmers.
    Answer only by listing the projects.
    '''
)

chat_template_time = ChatPromptTemplate.from_template(
    '''
    I'm an intermediate level programmer.
    
    Consider the following literature:
    {books}

    Also, consider the following projects:
    {projects}

    Roughly how much time would it take me to complete the literature and projects?
    '''
)

In [8]:
chat = ChatOpenAI(model_name = 'gpt-4', 
                  seed = 365,
                  temperature = 0,
                  max_tokens = 500)

In [9]:
string_parser = StrOutputParser()

In [10]:
chain_books = chat_template_books | chat | string_parser

chain_projects = chat_template_projects | chat | string_parser

In [11]:
chain_parallel = RunnableParallel({'books': chain_books, 'projects': chain_projects})

In [12]:
chain_parallel.invoke({'programming language': 'Python'})

{'books': '1. "Fluent Python: Clear, Concise, and Effective Programming" by Luciano Ramalho\n2. "Python Cookbook: Recipes for Mastering Python 3" by David Beazley and Brian K. Jones\n3. "Effective Python: 90 Specific Ways to Write Better Python" by Brett Slatkin',
 'projects': '1. Building a Web Scraper using BeautifulSoup and Requests.\n2. Developing a Text-Based Adventure Game using Object-Oriented Programming.\n3. Creating a Personal Finance Tracker with GUI using Tkinter.'}

In [16]:
chain_time1 = (RunnableParallel({'books':chain_books,
                                'projects':chain_projects})
             | chat_template_time
             | chat
             | string_parser
            )

In [20]:
chain_time2 = ({'books':chain_books,
                'projects':chain_projects}
             | chat_template_time
             | chat
             | string_parser
            )

In [22]:
print(chain_time2.invoke({'programming language': 'Python'}))

The time it takes to complete the literature and projects can vary greatly depending on several factors such as your reading speed, comprehension level, familiarity with the topics, and the amount of time you can dedicate each day. 

However, as a rough estimate:

1. "Fluent Python: Clear, Concise, and Effective Programming" - This book is around 800 pages. If you read and practice for about 2 hours a day, it might take you around 1-2 months to complete.

2. "Python Cookbook: Recipes for Mastering Python 3" - This book is around 700 pages. Again, if you read and practice for about 2 hours a day, it might take you around 1-2 months to complete.

3. "Effective Python: 90 Specific Ways to Write Better Python" - This book is around 230 pages. If you read and practice for about 2 hours a day, it might take you around 2-3 weeks to complete.

For the projects:

1. Building a Web Scraper using BeautifulSoup - If you're familiar with the basics of web scraping, this could take you a few days to

In [23]:
chain_time2.get_graph().print_ascii()

            +-------------------------------+              
            | Parallel<books,projects>Input |              
            +-------------------------------+              
                   ***               ***                   
                ***                     ***                
              **                           **              
+--------------------+              +--------------------+ 
| ChatPromptTemplate |              | ChatPromptTemplate | 
+--------------------+              +--------------------+ 
           *                                   *           
           *                                   *           
           *                                   *           
    +------------+                      +------------+     
    | ChatOpenAI |                      | ChatOpenAI |     
    +------------+                      +------------+     
           *                                   *           
           *                            